### C02 銘柄へのラベル付けプログラム
・L_高配当  
・L_優待  
・L_その他候補  
・L_所有  
・L_NG  

In [1]:
# (1) インポート_標準ライブラリ一括
import os
import csv
import re
import tempfile
from typing import Dict, List, Set, Tuple


# (2) 手動ラベル設定_対象リストと紐付け定義
# (2-1) 高配当_銘柄コード一覧
HIGH_DIV_LIST = [
    1414, 1723, 1835, 1928, 1951, 2003, 2124, 2169, 2296, 2353,
    2391, 2393, 2670, 2914, 3076, 3407, 3763, 3817, 3834, 4008,
    4041, 4042, 4220, 4326, 4452, 4502, 4641, 4658, 4732, 4762,
    4768, 4832, 5011, 5108, 5334, 5388, 5401, 5464, 5857, 6073,
    6087, 6113, 6301, 6322, 6454, 6539, 6745, 6750, 6919, 6957,
    7820, 7921, 7931, 7995, 8001, 8002, 8031, 8053, 8058, 8130,
    8306, 8316, 8473, 8584, 8591, 8593, 8630, 8750, 8766, 8898,
    9069, 9142, 9303, 9368, 9432, 9433, 9436, 9513, 9769, 9882,
    9960, 9986,
    1343, 1489, 1488, 2556, 2558,
    1976, 1980, 2185, 2269, 2317, 2374, 2659, 3231, 3333, 3771,
    4248, 4345, 4401, 4540, 4719, 4752, 4972, 5186, 6345, 6381,
    6432, 6458, 6652, 6678, 6785, 7292, 7438, 7483, 7723, 7749,
    7817, 7989, 7994, 9057, 9069, 9233, 9304, 9381, 9687,
]

# (2-2) 優待_銘柄コード一覧
PERK_LIST = [
    2157, 2379, 2493, 2678, 2702, 2722, 2752, 2796, 3053, 3140,
    3320, 3387, 3645, 3863, 3880, 3926, 4447, 4755, 4936, 6504,
    6630, 7213, 7296, 7388, 7698, 8079, 8095, 8613, 8904, 8917,
    9434, 9612
]

# (2-3) その他候補_銘柄コード一覧
OTHER_LIST: List[int] = []

# (2-4) NG_銘柄コード一覧
NG_LIST: List[int] = []

# (2-5) 更新対象ラベル定義（L_所有は除外）
TARGET_LABELS = ["L_高配当", "L_優待", "L_その他候補", "L_NG"]

# (2-6) ラベル名→コード一覧 の紐付け
LABEL_MAPPING: List[Tuple[str, List[int]]] = [
    ("L_高配当",     HIGH_DIV_LIST),
    ("L_優待",       PERK_LIST),
    ("L_その他候補", OTHER_LIST),
    ("L_NG",         NG_LIST),
]


# (3) 設定_入出力パス
IDMAP_CSV = os.path.join("Data", "01_IDmap.csv")


# (4) ユーティリティ_コード正規化と判定用セット作成
# (4-1) code正規化（数値/文字を想定）
def _norm_code(code) -> str:
    if code is None:
        return ""
    s = str(code).strip()
    # 先頭の数値だけ抜く（"8306" / "8306.T" / "8306 " などに強くする）
    m = re.match(r"^\d+", s)
    return m.group(0) if m else s


# (4-2) ラベル判定用のセットを作成（高速 membership）
def build_label_sets() -> Dict[str, Set[str]]:
    label_sets: Dict[str, Set[str]] = {}
    for label_name, code_list in LABEL_MAPPING:
        label_sets[label_name] = { _norm_code(c) for c in code_list if _norm_code(c) }
    return label_sets


# (5) メイン処理_CSVをストリーミングで更新（他列/他行は保持、ラベル列のみ確定）
# (5-1) CSV更新本体（目的：TARGET_LABELSのみ更新、列が無ければ作成、他は保持）
def update_labels_in_csv(in_csv: str, out_csv: str) -> None:
    if not os.path.exists(in_csv):
        raise FileNotFoundError(f"入力CSVが見つかりません: {in_csv}")

    label_sets = build_label_sets()

    out_dir = os.path.dirname(out_csv) or "."
    os.makedirs(out_dir, exist_ok=True)

    fd, tmp = tempfile.mkstemp(prefix="idmap_labels_", suffix=".csv", dir=out_dir)
    os.close(fd)

    try:
        with open(in_csv, "r", encoding="utf-8-sig", newline="") as fin:
            reader = csv.DictReader(fin)

            # 入力ヘッダを維持しつつ、TARGET_LABELSが無ければ追加する
            in_fields = list(reader.fieldnames or [])
            field_set = set(in_fields)

            out_fields = list(in_fields)
            for lbl in TARGET_LABELS:
                if lbl not in field_set:
                    out_fields.append(lbl)

            with open(tmp, "w", encoding="utf-8-sig", newline="") as fout:
                writer = csv.DictWriter(fout, fieldnames=out_fields, extrasaction="ignore")
                writer.writeheader()

                for row in reader:
                    # 行の他列はそのまま保持（row自体は必要最低限だけ変更）
                    code = _norm_code(row.get("code", ""))

                    # ラベル列のみ確定（追記ではなく、0/1の上書きで目的動作を固定）
                    for lbl in TARGET_LABELS:
                        s = label_sets.get(lbl, set())
                        row[lbl] = "1" if (code and code in s) else "0"

                    # そのまま書き出し（他列/他行を維持）
                    writer.writerow(row)

        os.replace(tmp, out_csv)
        print(f"保存完了: {out_csv}（ラベル列のみ更新、他列/他行は保持）")

    except Exception as e:
        # 失敗時は元ファイルを壊さない
        print(f"エラー: {e}")
        if os.path.exists(tmp):
            os.remove(tmp)
        raise


# (6) エントリポイント
def main() -> None:
    print(">>> ラベル列のみ更新します（他の列/行は消しません。列が無ければ作成します）")
    update_labels_in_csv(IDMAP_CSV, IDMAP_CSV)


# (7) 実行
if __name__ == "__main__":
    main()


>>> ラベル列のみ更新します（他の列/行は消しません。列が無ければ作成します）
保存完了: Data\01_IDmap.csv（ラベル列のみ更新、他列/他行は保持）


### 動作確認用

In [2]:
# (1) インポート_標準/外部ライブラリ
import os
import pandas as pd
from IPython.display import display


# (2) 設定_ファイルパスとラベル列
IDMAP_CSV = os.path.join("Data", "01_IDmap.csv")
LABEL_NAMES = ["L_高配当", "L_優待", "L_その他候補", "L_所有", "L_NG"]


# (3) 取得_ラベル付き銘柄の表示用DataFrameを作る
def get_labeled_df() -> pd.DataFrame | None:
    # (3-1) 入力チェック
    if not os.path.exists(IDMAP_CSV):
        print(f"エラー: {IDMAP_CSV} が見つかりません。")
        return None

    # (3-2) CSV読み込み（欠損は一旦空に）
    df = pd.read_csv(IDMAP_CSV, dtype=str, encoding="utf-8-sig").fillna("")

    # (3-3) 必須列の補完（無い列は 0 として追加）
    # ※ ラベル更新スクリプトが追加しない列があっても落ちないようにする
    for col in ["code", "name"]:
        if col not in df.columns:
            df[col] = ""

    for lbl in LABEL_NAMES:
        if lbl not in df.columns:
            df[lbl] = "0"

    # ラベル列の空/NaNは "0" 扱い
    df[LABEL_NAMES] = df[LABEL_NAMES].replace({"": "0"}).fillna("0")

    # (3-4) フィルタリング：いずれかのラベル列が "1" の行を抽出
    condition = (df[LABEL_NAMES] == "1").any(axis=1)
    df_filtered = df.loc[condition].copy()

    if df_filtered.empty:
        print("ラベル（1）が付与された銘柄は見つかりませんでした。")
        return None

    # (3-5) 整形：数値順でソート
    df_filtered["sort_key"] = pd.to_numeric(df_filtered["code"], errors="coerce")
    df_sorted = df_filtered.sort_values("sort_key").drop(columns=["sort_key"])

    # 表示する列の選択（基本情報 + ラベル列）
    display_cols = ["code", "name"] + LABEL_NAMES

    # 0を "-" に置換して視認性を高める
    view_df = df_sorted[display_cols].replace("0", "-")

    return view_df


# (4) 実行_Jupyterで表示
df_result = get_labeled_df()
df_result


,code,name,L_高配当,L_優待,L_その他候補,L_所有,L_NG
316,1343,ＮＥＸＴ ＦＵＮＤＳ 東証ＲＥＩＴ指数連動型上場投信,1,-,-,-,-
345,1414,ショーボンド HD,1,-,-,-,-
388,1488,ｉＦｒｅｅＥＴＦ 東証ＲＥＩＴ指数,1,-,-,-,-
389,1489,-,1,-,-,-,-
503,1723,日本電技,1,-,-,-,-
...,...,...,...,...,...,...,...
4286,9687,KSK,1,-,-,-,-
4327,9769,学究社,1,-,-,-,-
4368,9882,イエローハット,1,-,-,-,-
4398,9960,東テク,1,-,-,-,-


### コードチェック

In [3]:
import os
import pandas as pd

# (1) 設定
IDMAP_CSV = os.path.join("Data", "01_IDmap.csv")
LABEL_NAMES = ["L_高配当", "L_優待", "L_その他候補", "L_所有", "L_NG"]

def check_stock_label(target_code: str):
    if not os.path.exists(IDMAP_CSV):
        print(f"エラー: {IDMAP_CSV} が見つかりません。")
        return

    # CSV読み込み
    df = pd.read_csv(IDMAP_CSV, dtype=str, encoding="utf-8-sig").fillna("0")
    
    # 指定されたコードで検索
    row = df[df["code"] == str(target_code)]

    if row.empty:
        print(f"--- 検索結果 ---")
        print(f"銘柄コード [{target_code}] は見つかりませんでした。")
        return

    # データ抽出（1行目を取得）
    data = row.iloc[0]

    # 結果表示
    print(f"--- 銘柄ラベル確認 ---")
    print(f"【基本情報】")
    print(f"  コード: {data['code']}")
    print(f"  銘柄名: {data['name']}")
    print(f"  業種  : {data['industry_33']}")
    print(f"  URL   : {data['URL']}")
    print(f"  最終更新: {data['UPDATE']}")
    print(f"")
    print(f"【付与ラベル状況】")
    
    for lbl in LABEL_NAMES:
        # フラグが1なら [ON]、0なら [  ] と表示
        status = "★ [ON]" if data.get(lbl) == "1" else "[  ]"
        print(f"  {lbl.ljust(10)} : {status}")
    print(f"----------------------")

if __name__ == "__main__":
    # 調べたい銘柄コードをここに入力してください
    target = input("確認したい銘柄コード（4桁）を入力してください: ").strip()
    check_stock_label(target)

確認したい銘柄コード（4桁）を入力してください:  4228


--- 銘柄ラベル確認 ---
【基本情報】
  コード: 4228
  銘柄名: 積水化成品工業
  業種  : 化学
  URL   : https://irbank.net/E00845/results
  最終更新: 2025/12/21

【付与ラベル状況】
  L_高配当      : [  ]
  L_優待       : [  ]
  L_その他候補    : [  ]
  L_所有       : [  ]
  L_NG       : [  ]
----------------------
